# 🧠 ML 记忆唤醒训练场 (Memory Recovery Arena)

> **🎯 目标**: 消除你对 `get_dummies`、`train_test_split` 等 API 的陌生感。通过极其简单的填空，重新找回模型跑通的快感。
> 
> **📜 规则**: 
> 1. 我已经帮你写好了数据加载和探索。
> 2. 遇到 `🔍 你的代码：` 的地方，请打开你的速查手册 `18_ml_function_checklist` -> `🔥 核心：ML 黄金 Pipeline 傻瓜式代码流`，直接 Copy-Paste 并修改变量名即可。
> 3. 没有复杂的业务逻辑，没有特征工程，只关注 Pipeline 本身。

## 0. 准备工作：极简的流失数据集

In [1]:
import pandas as pd
import numpy as np

# 我们生成一个非常清晰、干净的数据集 (只有 4 列)
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age': np.random.randint(20, 60, size=n),
    'tenure_months': np.random.randint(1, 48, size=n),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n),
    'monthly_charges': np.random.uniform(20, 100, size=n)
})

# 制造一点规律：按月付款且花费高的年轻人容易流失
churn_prob = np.where((df['contract_type'] == 'Month-to-month') & (df['monthly_charges'] > 60), 0.7, 0.1)
df['churn'] = np.random.binomial(1, churn_prob)

df.head()

,age,tenure_months,contract_type,monthly_charges,churn
0,58,31,One year,99.017791,0
1,48,19,Two year,54.221851,0
2,34,47,Two year,50.746132,0
3,27,16,Month-to-month,74.371783,1
4,40,5,Month-to-month,37.460311,0


## 1. 特征编码 (One-Hot Encoding)
计算机看不懂 `contract_type` 里的 `Month-to-month`，把它变成数字吧！

In [11]:
# 🔍 你的代码：把 contract_type 列做 One-Hot 编码 (记住 drop_first=True)
# 参考速查手册：Step 1

df_encoded = pd.get_dummies(df, columns=['contract_type'], drop_first=True).astype(int)
df_encoded



,age,tenure_months,monthly_charges,churn,contract_type_One year,contract_type_Two year
0,58,31,99,0,1,0
1,48,19,54,0,0,1
2,34,47,50,0,0,1
3,27,16,74,1,0,0
4,40,5,37,0,0,0
...,...,...,...,...,...,...
495,31,6,56,0,1,0
496,28,6,66,0,0,0
497,26,3,33,0,0,0
498,47,7,78,1,0,0


## 2. 拆分特征(X)与标签(y)

In [13]:
# 🔍 你的代码：谁是目标特征？谁是预测依据？
# 参考速查手册：Step 2

X = df_encoded.drop(columns='churn')
y = df_encoded['churn']

## 3. 切分训练集与测试集 (Train-Test Split)
留作 20% 作为考试题。

In [14]:
from sklearn.model_selection import train_test_split

# 🔍 你的代码：切分 X 和 y (记住 stratify=y 保证流失率一致)
# 参考速查手册：Step 3

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

## 4. 召唤模型并预测 (Let's Go!)
不管以前多复杂，跑通这几行，你的信心就回来了。

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

# 🔍 你的代码：1.实例化模型，2.训练(fit)，3.硬预测(predict)，4.概率预测(predict_proba)
# 参考速查手册：Step 5

model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 告诉业务方：准不准？
print("---- 分类报告 ----")
print(classification_report(y_test, y_pred, target_names=['未流失', '流失']))

auc = roc_auc_score(y_test, y_pred_proba)
print(f"\n🔥 模型 AUC 得分: {auc:.4f}")

---- 分类报告 ----
              precision    recall  f1-score   support

         未流失       0.88      0.94      0.91        80
          流失       0.67      0.50      0.57        20

    accuracy                           0.85       100
   macro avg       0.77      0.72      0.74       100
weighted avg       0.84      0.85      0.84       100


🔥 模型 AUC 得分: 0.7819


**🎉 通关啦！** 现在，你可以深呼吸一下，看着跑出来的分数量化你的成就感。然后，随时准备带着这份肌肉记忆，杀回那个复杂的 #22 马拉松 Notebook 吧！